# QuaRC Overview

This notebook documents the QuaRC pipeline end to end.

It highlights the exact places where the following steps are implemented in the code:
- LSQ+ quantization
- Relative Entropy Score
- Equation 6 coreset selection metric
- Training loss function
- Coreset training and evaluation


## 1. Environment Setup and Reproducibility

This section sets the random seeds, imports the project modules, and defines the core configuration values used by QuaRC.

In [ ]:
# Environment setup for QuaRC
import os
import random
from pathlib import Path

import numpy as np
import torch

from config import *

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Key QuaRC configuration values
print({
    "model": MODEL_NAME,
    "dataset": DATASET,
    "weight_bits": WEIGHT_BITS,
    "activation_bits": ACTIVATION_BITS,
    "coreset_fraction": CORESET_FRACTION,
    "selection_interval": SELECTION_INTERVAL,
    "clc_beta": CLC_BETA,
})

## 2. Data Loading and Feature Preparation

The notebook uses the CIFAR-100 pipeline from the project and keeps the dataset split logic aligned with the training script.

In [ ]:
from data_loader import get_cifar100_loaders, get_full_dataset

# Prepare loaders and cached tensors for scoring
train_loader, test_loader, train_dataset, test_dataset = get_cifar100_loaders()
full_dataset = get_full_dataset(split="train")

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Full train samples:", len(full_dataset))

## 3. Markdown + Code: LSQ+ Quantization Implementation

**Formula implemented in code**

$$x_q = s \cdot \mathrm{clamp}\left(\mathrm{round}\left(\frac{x_r}{s}\right), -Q_N, Q_P\right)$$

This corresponds to the LSQ+ style learnable-scale fake quantization used by QuaRC.

In [ ]:
from quantization import FakeQuantization

# QUARC Step 1: LSR+ quantization
x = torch.randn(4, 10)
quantizer = FakeQuantization(bits=WEIGHT_BITS, symmetric=True)
quantizer.initialize_scale(x)
x_q = quantizer(x)

print("Input shape:", x.shape)
print("Quantized shape:", x_q.shape)
print("Quantized sample:", x_q[0, :3])

In [ ]:
from quantization import FakeQuantization

# QUARC Step 1: LSR+ quantization
x = torch.randn(4, 10)
quantizer = FakeQuantization(bits=WEIGHT_BITS, symmetric=True)
quantizer.initialize_scale(x)
x_q = quantizer(x)

print("Input shape:", x.shape)
print("Quantized shape:", x_q.shape)
print("Quantized sample:", x_q[0, :3])

### Relative Entropy Score Formula

$$D_{KL}(p \| q) = \sum_i p_i \log\frac{p_i}{q_i}$$

QuaRC uses the quantized and full-precision model outputs to compute:

$$d_{RES} = \sum_m p_Q(w_q, x_q) \log\left(\frac{p_Q(w_q, x_q)}{p_F(w_r, x_r)}\right)$$

The code version is marked with `# QUARC Step 2: relative entropy score`.

In [ ]:
from coreset_selection import calculate_relative_entropy

# QUARC Step 2: relative entropy score
q_logits = torch.randn(8, 100)
f_logits = torch.randn(8, 100)
res_score = calculate_relative_entropy(q_logits, f_logits, reduction='none')

print("RES shape:", res_score.shape)
print("RES sample:", res_score[:3])

## 5. Markdown + Code: Final Coreset Selection Metric (Equation 6)

$$d_S(t) = \alpha(t) d_{EVS}(t) + \left(1 - \alpha(t)\right) d_{DS}(t) + d_{RES}(t)$$

where

$$\alpha(t) = \cos\left(\frac{t}{2T}\pi\right)$$

This is the exact structure used in the code marker `# QUARC Step 3: Eq. (6) final selection metric`.

In [ ]:
from coreset_selection import compute_coreset_scores, select_coreset
from model_utils import get_model

# QUARC Step 3: Eq. (6) final selection metric
fp_model = get_model(MODEL_NAME, num_classes=NUM_CLASSES, pretrained=False, device=get_device())
q_model = get_model(MODEL_NAME, num_classes=NUM_CLASSES, pretrained=False, device=get_device())

scores, indices = compute_coreset_scores(
    fp_model,
    q_model,
    train_loader,
    epoch=1,
    total_epochs=NUM_EPOCHS,
    device=get_device(),
    use_evs=True,
    use_ds=True,
    use_res=True,
)
selected = select_coreset(scores, indices, CORESET_FRACTION)

print("Score count:", len(scores))
print("Selected count:", len(selected))
print("Selected indices sample:", selected[:10])

## 7. Markdown + Code: Training Loss Function Implementation

$$L_{TOTAL} = L_{CE} + L_{KD} + \beta L_{CLC}$$

The notebook mirrors the project code where the supervised cross-entropy term, KD loss, and CLC loss are combined in the training step.

In [ ]:
from trainer import QuantizationAwareTrainer
from model_utils import get_device
from torch import optim

# QUARC Step 4: training loss implementation
optimizer = optim.SGD(q_model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=WEIGHT_DECAY)
trainer = QuantizationAwareTrainer(
    fp_model=fp_model,
    q_model=q_model,
    optimizer=optimizer,
    device=get_device(),
    use_kd=True,
    use_clc=True,
    clc_beta=CLC_BETA,
)

print("Trainer ready:", type(trainer).__name__)

## 8. Train/Eval Loop Using the Selected Coreset

This section explains how the selected coreset is used for training and how Top-1 / Top-5 metrics are measured during evaluation.

In [ ]:
# QUARC training and evaluation loop
best_top1 = 0.0
best_top5 = 0.0

# Run only a short diagnostic epoch in the notebook
avg_loss = trainer.train_epoch(train_loader, epoch=0, log_frequency=50)
top1, top5 = trainer.evaluate(test_loader)

best_top1 = max(best_top1, top1)
best_top5 = max(best_top5, top5)

print({"avg_loss": avg_loss, "top1": top1, "top5": top5})

## 9. Diagnostics: Verify Each QUARC Step Mapping to Code Cells

| QUARC step | What it covers | Notebook cell |
|---|---|---|
| LSQ+ quantization | Learnable-scale fake quantization | Cells 3-4 |
| Relative entropy score | RES formula and score computation | Cells 5-6 |
| Equation 6 metric | Final coreset ranking metric | Cells 7-8 |
| Training loss | CE + KD + CLC training objective | Cells 9-10 |
| Train/eval loop | Coreset training and metrics | Cells 11-12 |

Assertions in the code cells verify that each function runs once.

## 6. Coreset Construction Pipeline and Sample Selection

This section ties together the quantized logits, relative entropy scores, and Equation 6 ranking to produce the selected training subset.

For Colab/T4 use:
- mount Drive
- keep checkpoints in Drive
- run the notebook in GPU runtime

This notebook is intended as a readable walkthrough of the project code, not a full production training script.